# M3 historical CLV edge-allocation failure diagnostic

이 노트북은 이미 완료된 seed 42 M1/M3 체크포인트를 다시 불러와 실패 원인을 설명합니다. 재학습, checkpoint 선택, 최종 test 및 holdout 생성은 하지 않습니다.

분석 질문은 네 가지입니다. (1) 엣지 개입이 실제로 충분히 컸는가, (2) 정답이 어느 순위 경계에서 이동했는가, (3) M3가 어떤 상품을 승격했는가, (4) 신규 정답에는 공동구매·순차구매·공통구매자·카테고리 중 어떤 train-only 연결이 남아 있는가.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys

REPO_DIR = '/content/clv-m2-lightgcn-runner'
SOURCE_REVISION = '84d2e2ada55ac4c1470332e06eb25a5dcea9a4fe'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run([
        'git', 'clone', '--branch', 'feat/m2-joint-nv-lightgcn',
        'https://github.com/jung-un/clv-m2-lightgcn-runner.git', REPO_DIR
    ], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'feat/m2-joint-nv-lightgcn'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--detach', SOURCE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_REVISION
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Pinned source:', SOURCE_REVISION)

## 1. 보호조건 확인

분석 대상은 `DAY 1~683` 학습, `DAY 684~690` 역사적 평가 결과뿐입니다.

In [ ]:
import json
from lightgcn_clv_m3_edge_allocation_diagnostic import (
    configure_m3_edge_allocation_diagnostic,
    preflight_summary,
    run_m3_edge_allocation_diagnostic,
)

OUT_DIR = '/content/drive/MyDrive/논문/data/results_m3_clv_edge_allocation_historical_dunnhumby'
cfg = configure_m3_edge_allocation_diagnostic(out_dir=OUT_DIR)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

## 2. 체크포인트 기반 진단 실행

Top-100 추천 복원과 train-only 연결성 계산 때문에 몇 분이 걸릴 수 있습니다. 모델 학습은 수행하지 않습니다.

In [ ]:
structure_evidence = run_m3_edge_allocation_diagnostic(cfg)
assert structure_evidence.attrs['quality_passed']
RESULT_PATHS = structure_evidence.attrs['result_paths']
print(json.dumps(RESULT_PATHS, ensure_ascii=False, indent=2))

## 3. 핵심 결과 불러오기

In [ ]:
import pandas as pd
from IPython.display import display

tables = {name: pd.read_csv(path) for name, path in RESULT_PATHS.items() if path.endswith('.csv')}
display(tables['quality_checks'])
display(tables['intervention_summary'])
display(tables['rank_movement_summary'])
display(tables['next_structure_evidence'].sort_values(['cutoff', 'truth_minus_m3_only'], ascending=[True, False]))

## 4. Top-10 하락과 Top-50 상승의 실제 이동량

In [ ]:
import matplotlib.pyplot as plt

rank_move = tables['rank_movement_summary'].set_index('cutoff')
ax = rank_move[['entered', 'left']].plot(kind='bar', figsize=(8, 4), color=['#2A9D8F', '#E76F51'])
ax.set_title('Truth entries and exits by rank cutoff | historical seed 42')
ax.set_xlabel('Rank cutoff')
ax.set_ylabel('Number of new-item truths')
ax.axhline(0, color='black', linewidth=0.7)
plt.tight_layout()
plt.show()

## 5. 어떤 그래프 관계가 정답에 더 강한가

양수는 해당 train-only 연결성이 M3가 새로 넣은 오답 상품보다 실제 신규 정답에서 더 강했다는 뜻입니다. 이는 다음 구조의 후보를 고르는 탐색 근거이며 인과효과가 아닙니다.

In [ ]:
evidence = tables['next_structure_evidence'].copy()
labels = {
    'shared_buyer_reach': '공통 구매자',
    'co_basket_reach': '공동 장바구니',
    'forward_transition_reach': '순차 구매',
    'history_category_share': '사용자 카테고리',
}
evidence['signal_label'] = evidence.signal.map(labels)
pivot = evidence.pivot(index='signal_label', columns='cutoff', values='truth_minus_m3_only')
ax = pivot.plot(kind='barh', figsize=(9, 5))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Train-only connectivity: truth minus M3-only recommendations')
ax.set_xlabel('Mean connectivity difference')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 6. CLV 및 N/V 사용자 집단별 차이

In [ ]:
segments = tables['segment_metric_summary']
focus = segments[segments.metric.isin([
    'recall@10', 'ndcg@10', 'recall@50',
    'price_purchase_amount_weighted_hit@10'
])].copy()
display(focus.sort_values(['segment_type', 'segment_id', 'metric']))
for segment_type in ['clv_quintile', 'nv_quadrant']:
    chart = focus[(focus.segment_type == segment_type) & focus.metric.isin(['recall@10', 'ndcg@10', 'recall@50'])]
    pivot = chart.pivot(index='segment_id', columns='metric', values='relative_change_pct')
    ax = pivot.plot(kind='bar', figsize=(10, 4))
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'M3 relative change by {segment_type} | one-seed descriptive')
    ax.set_ylabel('% versus M1')
    plt.tight_layout()
    plt.show()

## 7. M3 승격을 설명한 상품·엣지 특성

In [ ]:
correlations = tables['item_mechanism_correlations']
display(correlations.sort_values(['target', 'spearman'], ascending=[True, False]))
items = tables['item_mechanism']
display(items.sort_values(['top10_promotion_count', 'top10_promoted_hit_count'], ascending=False).head(30))

## 8. 대표 개선·악화 사용자와 실제 정답/추천상품

In [ ]:
display(tables['representative_users'])
details = tables['representative_user_details']
display(details.sort_values(['selection', 'user_idx', 'detail_role', 'rank'], na_position='first').head(300))

## 9. 다음 M3 구조를 고르는 판독표

- 상품별 coefficient ratio 분산이 거의 0이고 Kish 비율이 1에 가까우면: 실제 개입이 약한 문제입니다.
- 개입은 충분하지만 승격과 CLV 배분·관계강도의 상관이 0에 가까우면: 현재 엣지 신호 방향이 잘못된 문제입니다.
- 신규 정답이 M3-only 상품보다 공동 장바구니 또는 순차 구매 연결에서 뚜렷하게 강하면: 다음 M3는 train-only 상품–상품 엣지를 검토할 근거가 됩니다.
- 카테고리 연결만 강하면: 카테고리 노드를 포함한 이종 그래프가 더 직접적인 후보입니다.
- Top-50 순진입은 양수지만 Top-10 순진입이 0 이하라면: 후보 도달성보다 상위 정렬 문제가 큽니다. 그래프 확장만으로 해결된다고 단정하지 않습니다.

이 판독은 구조 후보를 만드는 사후 분석입니다. 같은 역사적 평가구간에서 새 구조의 성공을 다시 확증하지 않습니다.